# 🔮 TelecomX — Parte 2: Modelos Predictivos de Churn
## Challenge Data Science LATAM 2026 | Alura ONE

---

| | |
|---|---|
| **Empresa** | TelecomX |
| **Objetivo** | Predecir qué clientes tienen mayor probabilidad de cancelar el servicio |
| **Dataset** | TelecomX_Data.json (resultado de la limpieza — Parte 1) |
| **Técnica** | Clasificación supervisada con ML |

---

### 📋 Estructura del Notebook

| Sección | Descripción |
|---------|-------------|
| **1. Preparación de datos** | Carga, encoding, balanceo, normalización |
| **2. Correlación y selección** | Matriz de correlación, boxplots, scatter plots |
| **3. Modelado predictivo** | Split, entrenamiento y evaluación de modelos |
| **4. Interpretación y conclusiones** | Importancia de variables, estrategias de retención |


## 📦 Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, accuracy_score, f1_score,
    precision_score, recall_score
)

sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.dpi'] = 110

print("✅ Librerías cargadas correctamente")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")
print(f"   sklearn : {__import__('sklearn').__version__}")


---
## 🗂️ Sección 1 — Preparación de los Datos

Esta sección cubre los pasos de:
- **1A** — Extracción del archivo tratado (Parte 1)
- **1B** — Eliminación de columnas irrelevantes
- **1C** — Encoding de variables categóricas
- **1D** — Verificación de la proporción de Churn
- **1E** — Balanceo de clases (oversampling manual)
- **1F** — Normalización / Estandarización


### 1A — Extracción del Archivo Tratado

Cargamos el archivo `TelecomX_Data.json` que contiene los datos ya **limpios y estandarizados** de la Parte 1.  
Este archivo incluye solo columnas relevantes con datos corregidos.


In [ ]:
import urllib.request, json as _json
import pandas as pd

URL = "https://raw.githubusercontent.com/michgdigital1/challengetelecomx2/refs/heads/main/TelecomX_Data_Part2.json"

try:
    with urllib.request.urlopen(URL, timeout=10) as r:
        raw = _json.loads(r.read().decode())
    print("✅ Datos cargados desde GitHub")
except Exception as e:
    print(f"⚠️  Sin conexión ({e}). Usando dataset embebido representativo...")
    raw = None

# ── Aplanar estructura anidada ────────────────────────────────────────────────
def aplanar(registro):
    """Desanida los sub-objetos customer, phone, internet, account, Charges."""
    fila = {}
    fila['customerID']       = registro.get('customerID')
    fila['Churn']            = registro.get('Churn')
    # Sub-objeto customer
    for k, v in registro.get('customer', {}).items():
        fila[k] = v
    # Sub-objeto phone
    for k, v in registro.get('phone', {}).items():
        fila[k] = v
    # Sub-objeto internet
    for k, v in registro.get('internet', {}).items():
        fila[k] = v
    # Sub-objeto account
    account = registro.get('account', {})
    fila['Contract']         = account.get('Contract')
    fila['PaperlessBilling'] = account.get('PaperlessBilling')
    fila['PaymentMethod']    = account.get('PaymentMethod')
    # Sub-objeto Charges (dentro de account)
    charges = account.get('Charges', {})
    fila['MonthlyCharges']   = charges.get('Monthly')
    fila['TotalCharges']     = charges.get('Total')
    return fila

if raw is not None:
    df_raw = pd.DataFrame([aplanar(r) for r in raw])
    print(f"✅ Estructura aplanada correctamente")
else:
    # Dataset de respaldo
    import numpy as np
    np.random.seed(42)
    n = 7043
    contrato  = np.random.choice(['Month-to-month','One year','Two year'], n, p=[0.55,0.24,0.21])
    tenure    = np.where(contrato=='Month-to-month', np.random.randint(1,24,n),
                np.where(contrato=='One year', np.random.randint(12,48,n), np.random.randint(24,72,n)))
    monthly   = np.where(contrato=='Month-to-month', np.random.uniform(45,110,n),
                np.where(contrato=='One year', np.random.uniform(40,95,n), np.random.uniform(35,90,n)))
    total     = (tenure * monthly * np.random.uniform(0.85,1.05,n)).round(2)
    p_churn   = (0.38*(contrato=='Month-to-month')+0.09*(contrato=='One year')+
                 0.03*(contrato=='Two year')+0.14*(monthly>80)+0.10*(tenure<6)-
                 0.05*(tenure>48)+np.random.normal(0,0.05,n)).clip(0.02,0.95)
    internet  = np.random.choice(['DSL','Fiber optic','No'], n, p=[0.34,0.44,0.22])
    def dep(arr, p=0.34):
        return ['No internet service' if s=='No' else np.random.choice(['Yes','No'],p=[p,1-p]) for s in arr]
    df_raw = pd.DataFrame({
        'customerID':       [f'TC{i:05d}' for i in range(n)],
        'gender':           np.random.choice(['Male','Female'], n),
        'SeniorCitizen':    np.random.choice([0,1], n, p=[0.84,0.16]),
        'Partner':          np.random.choice(['Yes','No'], n, p=[0.48,0.52]),
        'Dependents':       np.random.choice(['Yes','No'], n, p=[0.30,0.70]),
        'tenure':           tenure,
        'PhoneService':     np.random.choice(['Yes','No'], n, p=[0.90,0.10]),
        'MultipleLines':    np.random.choice(['Yes','No','No phone service'], n, p=[0.42,0.48,0.10]),
        'InternetService':  internet,
        'OnlineSecurity':   dep(internet,0.29),
        'OnlineBackup':     dep(internet,0.34),
        'DeviceProtection': dep(internet,0.34),
        'TechSupport':      dep(internet,0.29),
        'StreamingTV':      dep(internet,0.38),
        'StreamingMovies':  dep(internet,0.39),
        'Contract':         contrato,
        'PaperlessBilling': np.random.choice(['Yes','No'], n, p=[0.59,0.41]),
        'PaymentMethod':    np.random.choice(['Electronic check','Mailed check',
                              'Bank transfer (automatic)','Credit card (automatic)'], n, p=[0.34,0.23,0.22,0.21]),
        'MonthlyCharges':   monthly.round(2),
        'TotalCharges':     total,
        'Churn':            np.where(np.random.binomial(1,p_churn)==1,'Yes','No')
    })

print(f"\n📊 Shape: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")
print(f"   Columnas: {list(df_raw.columns)}")
df_raw.head(3)


### 1B — Eliminación de Columnas Irrelevantes

Eliminamos columnas que **no aportan valor predictivo**, como el identificador único del cliente (`customerID`). Este tipo de variable no tiene relación con el comportamiento de cancelación y puede generar ruido en los modelos.


In [ ]:
df = df_raw.copy()

# ── Convertir TotalCharges a numérico (viene como string en el JSON original) ─
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
nulos_tc = df['TotalCharges'].isnull().sum()
if nulos_tc > 0:
    mediana_tc = df['TotalCharges'].median()
    df['TotalCharges'].fillna(mediana_tc, inplace=True)
    print(f"  ↳ {nulos_tc} nulos en TotalCharges imputados con mediana ({mediana_tc:.2f})")
else:
    print("  ↳ TotalCharges sin valores nulos")

# ── Verificar y convertir MonthlyCharges ──────────────────────────────────────
df['MonthlyCharges'] = pd.to_numeric(df['MonthlyCharges'], errors='coerce')

# ── Eliminar customerID ───────────────────────────────────────────────────────
cols_irrelevantes = [c for c in ['customerID'] if c in df.columns]
df.drop(columns=cols_irrelevantes, inplace=True)

print(f"✅ Columnas eliminadas: {cols_irrelevantes}")
print(f"   Shape resultante   : {df.shape}")
print(f"\n📋 Columnas disponibles ({len(df.columns)}):")
print(list(df.columns))
df.head(3)


### 1C — Encoding de Variables Categóricas

Transformamos las variables categóricas a formato numérico usando **One-Hot Encoding** (`pd.get_dummies`).  
Esto crea columnas binarias por cada categoría, haciéndolas compatibles con los algoritmos de ML.

> La variable objetivo `Churn` se codifica por separado: `Yes → 1`, `No → 0`.


In [ ]:
# Codificar variable objetivo
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Identificar variables categóricas (excluyendo Churn)
cat_cols = df.select_dtypes(include='object').columns.tolist()
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols = [c for c in num_cols if c != 'Churn']

print(f"Variables categóricas a codificar ({len(cat_cols)}): {cat_cols}")
print(f"Variables numéricas ({len(num_cols)}): {num_cols}")

# One-Hot Encoding
df_enc = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print(f"\n✅ One-Hot Encoding aplicado")
print(f"   Shape antes  : {df.shape}")
print(f"   Shape después: {df_enc.shape}")
print(f"   Nuevas columnas generadas: {df_enc.shape[1] - df.shape[1] + len(cat_cols)}")
df_enc.head(3)


### 1D — Verificación de la Proporción de Churn

Antes de entrenar modelos, es fundamental evaluar si existe **desbalance entre las clases**.  
Un fuerte desbalance puede sesgar al modelo a predecir siempre la clase mayoritaria.


In [ ]:
# Proporción con value_counts()
churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100

proporcion = pd.DataFrame({
    'Cantidad'       : churn_counts,
    'Porcentaje (%)' : churn_pct.round(2)
})
proporcion.index = ['No Churn (0)', 'Churn (1)']
print("=" * 40)
print("  PROPORCIÓN DE CHURN (value_counts)")
print("=" * 40)
print(proporcion.to_string())
print("=" * 40)

ratio = churn_counts[0] / churn_counts[1]
print(f"\n⚠️  Ratio de desbalance: {ratio:.1f}:1 (No Churn : Churn)")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colores = ['#2ECC71', '#E74C3C']

churn_counts.plot(kind='bar', ax=axes[0], color=colores, edgecolor='white', width=0.5)
axes[0].set_title('Conteo por Clase', fontweight='bold')
axes[0].set_xticklabels(['No Churn', 'Churn'], rotation=0)
axes[0].set_ylabel('Número de Clientes')
for bar in axes[0].patches:
    axes[0].annotate(f'{int(bar.get_height()):,}',
                     (bar.get_x() + bar.get_width()/2, bar.get_height()),
                     ha='center', va='bottom', fontweight='bold')

axes[1].pie(churn_counts.values, labels=['No Churn', 'Churn'], colors=colores,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize':11})
axes[1].set_title('Proporción de Clases', fontweight='bold')

plt.suptitle('1D — Distribución de la Variable Objetivo: Churn', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📌 Conclusión: El dataset presenta desbalance de clases.")
print("   La clase minoritaria (Churn=1) representa ~26% del total.")
print("   Se aplicará balanceo en el siguiente paso.")


### 1E — Balanceo de Clases (Oversampling)

Dado el desbalance detectado (~74% No Churn / ~26% Churn), aplicamos **oversampling manual** de la clase minoritaria (Churn=1) para equilibrar las clases.

> **Nota:** Si tienes `imbalanced-learn` instalado, puedes usar `SMOTE` para generar ejemplos sintéticos más sofisticados. Aquí utilizamos resample de sklearn, disponible en cualquier entorno.


In [ ]:
from sklearn.utils import resample

# Separar clases
df_mayoria  = df_enc[df_enc['Churn'] == 0]
df_minoria  = df_enc[df_enc['Churn'] == 1]

print(f"Antes del balanceo:")
print(f"  No Churn (0): {len(df_mayoria):,}")
print(f"  Churn    (1): {len(df_minoria):,}")

# Oversampling de la clase minoritaria
df_minoria_over = resample(
    df_minoria,
    replace=True,
    n_samples=len(df_mayoria),
    random_state=42
)

df_balanced = pd.concat([df_mayoria, df_minoria_over]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nDespués del balanceo (oversampling):")
print(f"  No Churn (0): {(df_balanced['Churn']==0).sum():,}")
print(f"  Churn    (1): {(df_balanced['Churn']==1).sum():,}")
print(f"  Total       : {len(df_balanced):,}")

# Visualización comparativa
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colores = ['#2ECC71', '#E74C3C']

df_enc['Churn'].value_counts().plot(kind='bar', ax=axes[0], color=colores, edgecolor='white', width=0.5)
axes[0].set_title('Antes del Balanceo', fontweight='bold')
axes[0].set_xticklabels(['No Churn', 'Churn'], rotation=0)
axes[0].set_ylabel('Clientes')
for b in axes[0].patches:
    axes[0].annotate(f'{int(b.get_height()):,}', (b.get_x()+b.get_width()/2, b.get_height()),
                     ha='center', va='bottom', fontweight='bold')

df_balanced['Churn'].value_counts().plot(kind='bar', ax=axes[1], color=colores, edgecolor='white', width=0.5)
axes[1].set_title('Después del Balanceo (Oversampling)', fontweight='bold')
axes[1].set_xticklabels(['No Churn', 'Churn'], rotation=0)
for b in axes[1].patches:
    axes[1].annotate(f'{int(b.get_height()):,}', (b.get_x()+b.get_width()/2, b.get_height()),
                     ha='center', va='bottom', fontweight='bold')

plt.suptitle('1E — Balanceo de Clases', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


### 1F — Normalización / Estandarización

La necesidad de normalizar depende del tipo de modelo:

| Requiere normalización | No requiere normalización |
|---|---|
| Regresión Logística ✅ | Árbol de Decisión ❌ |
| KNN ✅ | Random Forest ❌ |
| SVM ✅ | Gradient Boosting ❌ |

**Estrategia:** Crearemos **dos versiones** del dataset de entrenamiento:
- `X_train_scaled` / `X_test_scaled` → para modelos sensibles a escala
- `X_train` / `X_test` → para modelos basados en árboles


In [ ]:
# ── Separar features y target desde dataset balanceado ───────────────────────
X = df_balanced.drop(columns=['Churn'])
y = df_balanced['Churn']

# ── División 80/20 con estratificación ────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✅ División Train/Test (80/20)")
print(f"   Train : {X_train.shape[0]:,} muestras  |  Churn rate: {y_train.mean()*100:.1f}%")
print(f"   Test  : {X_test.shape[0]:,} muestras   |  Churn rate: {y_test.mean()*100:.1f}%")

# ── StandardScaler sobre variables numéricas ─────────────────────────────────
num_features = [c for c in num_cols if c in X_train.columns]
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[num_features] = scaler.fit_transform(X_train[num_features])
X_test_scaled[num_features]  = scaler.transform(X_test[num_features])

print(f"\n✅ StandardScaler aplicado a variables numéricas: {num_features}")
print(f"   Media (train) después del escalado ≈ 0")
print(f"   Std  (train) después del escalado ≈ 1")
print(f"\n📌 Listo: X_train / X_test          → para modelos basados en árboles")
print(f"          X_train_scaled / X_test_scaled → para Regresión Logística y KNN")


---
## 📊 Sección 2 — Correlación y Selección de Variables

- **2A** — Matriz de correlación
- **2B** — Análisis dirigido: tenure × Churn y TotalCharges × Churn


### 2A — Análisis de Correlación

Visualizamos la **matriz de correlación** para identificar relaciones entre variables numéricas, con especial atención a aquellas con mayor correlación con `Churn`.


In [ ]:
# Correlación sobre el dataset pre-balanceo (valores originales, más interpretable)
df_corr = df_enc.copy()

corr_matrix = df_corr.corr()
corr_churn  = corr_matrix['Churn'].drop('Churn').sort_values(key=abs, ascending=False)

print("🔺 TOP 10 — Correlación POSITIVA con Churn:")
print(corr_churn[corr_churn > 0].head(10).round(4).to_string())
print("\n🔻 TOP 10 — Correlación NEGATIVA con Churn:")
print(corr_churn[corr_churn < 0].head(10).round(4).to_string())


In [ ]:
# ── Heatmap de correlación (Top 16 variables + Churn) ────────────────────────
top_vars = corr_churn.abs().head(15).index.tolist() + ['Churn']
corr_sub  = df_corr[top_vars].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_sub, dtype=bool))
sns.heatmap(corr_sub, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.75}, annot_kws={'size': 8})
plt.title('2A — Matriz de Correlación: Top 15 Variables vs Churn',
          fontsize=13, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# ── Barplot: correlación de TODAS las variables con Churn ────────────────────
top20_vars = corr_churn.abs().head(20).index
vals_plot  = corr_churn[top20_vars]
bar_colors = ['#E74C3C' if v > 0 else '#3498DB' for v in vals_plot.values]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(range(len(vals_plot)), vals_plot.values,
               color=bar_colors, edgecolor='white', height=0.7)
ax.set_yticks(range(len(vals_plot)))
ax.set_yticklabels(vals_plot.index, fontsize=9)
ax.axvline(0, color='gray', lw=0.8, linestyle='--')
ax.set_xlabel('Correlación con Churn', fontsize=11)
ax.set_title('Top 20 Variables — Correlación con Churn\n🔴 Positiva (aumenta riesgo)  |  🔵 Negativa (reduce riesgo)',
             fontsize=12, fontweight='bold')
for bar, val in zip(bars, vals_plot.values):
    ax.text(val + 0.002 if val >= 0 else val - 0.002,
            bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=8)
plt.tight_layout()
plt.show()


### 2B — Análisis Dirigido

Investigamos la relación entre variables específicas y la cancelación usando **boxplots** y **scatter plots**.


In [ ]:
# ── 2B.1 Tiempo de Contrato (tenure) × Cancelación — Boxplot ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

df_plot = df.copy()
df_plot['Churn_label'] = df_plot['Churn'].map({1: 'Churn', 0: 'No Churn'})

# Boxplot tenure
sns.boxplot(data=df_plot, x='Churn_label', y='tenure',
            palette={'No Churn': '#2ECC71', 'Churn': '#E74C3C'},
            ax=axes[0], width=0.5)
axes[0].set_title('Antigüedad (tenure) × Cancelación', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Estado del Cliente')
axes[0].set_ylabel('Meses como Cliente (tenure)')

# Agregar medias
for i, grupo in enumerate(['No Churn', 'Churn']):
    media = df_plot[df_plot['Churn_label'] == grupo]['tenure'].mean()
    axes[0].text(i, media + 1, f'μ={media:.1f}', ha='center',
                 fontsize=10, color='black', fontweight='bold')

# Boxplot TotalCharges
sns.boxplot(data=df_plot, x='Churn_label', y='TotalCharges',
            palette={'No Churn': '#2ECC71', 'Churn': '#E74C3C'},
            ax=axes[1], width=0.5)
axes[1].set_title('Gasto Total (TotalCharges) × Cancelación', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Estado del Cliente')
axes[1].set_ylabel('Gasto Total ($)')

for i, grupo in enumerate(['No Churn', 'Churn']):
    media = df_plot[df_plot['Churn_label'] == grupo]['TotalCharges'].mean()
    axes[1].text(i, media + 50, f'μ=${media:,.0f}', ha='center',
                 fontsize=10, color='black', fontweight='bold')

plt.suptitle('2B — Análisis Dirigido: Variables Clave vs Churn (Boxplots)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Estadísticas
print("📊 Estadísticas por grupo:")
for var in ['tenure', 'TotalCharges', 'MonthlyCharges']:
    stats = df_plot.groupby('Churn_label')[var].agg(['mean','median','std']).round(2)
    print(f"\n  {var}:")
    print(stats.to_string())


In [ ]:
# ── 2B.2 Scatter plot: tenure vs TotalCharges coloreado por Churn ────────────
muestra = df_plot.sample(min(2000, len(df_plot)), random_state=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter 1: tenure vs TotalCharges
for label, color in [('No Churn', '#2ECC71'), ('Churn', '#E74C3C')]:
    sub = muestra[muestra['Churn_label'] == label]
    axes[0].scatter(sub['tenure'], sub['TotalCharges'],
                    c=color, label=label, alpha=0.4, s=15)
axes[0].set_xlabel('Antigüedad (meses)', fontsize=11)
axes[0].set_ylabel('Gasto Total ($)', fontsize=11)
axes[0].set_title('tenure vs TotalCharges', fontweight='bold')
axes[0].legend()

# Scatter 2: tenure vs MonthlyCharges
for label, color in [('No Churn', '#2ECC71'), ('Churn', '#E74C3C')]:
    sub = muestra[muestra['Churn_label'] == label]
    axes[1].scatter(sub['tenure'], sub['MonthlyCharges'],
                    c=color, label=label, alpha=0.4, s=15)
axes[1].set_xlabel('Antigüedad (meses)', fontsize=11)
axes[1].set_ylabel('Cargo Mensual ($)', fontsize=11)
axes[1].set_title('tenure vs MonthlyCharges', fontweight='bold')
axes[1].legend()

plt.suptitle('2B — Scatter Plots: Patrones de Cancelación',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📌 Observaciones:")
print("  • Clientes con BAJA antigüedad y ALTO cargo mensual tienen mayor riesgo de Churn.")
print("  • Clientes con ALTA antigüedad acumulan más gasto total (son los más leales).")


---
## 🤖 Sección 3 — Modelado Predictivo

- **3A** — Separación de datos (80/20, ya realizada en 1F)
- **3B** — Creación de modelos (con y sin normalización)
- **3C** — Evaluación y comparación de modelos


### 3A — Separación de Datos

La división fue realizada en la **Sección 1F** con estratificación para preservar la proporción de clases.  
Recordatorio de los conjuntos disponibles:

| Conjunto | Uso |
|---|---|
| `X_train` / `X_test` | Árbol de Decisión, Random Forest |
| `X_train_scaled` / `X_test_scaled` | Regresión Logística, KNN |


In [ ]:
print("✅ División ya realizada en Sección 1F:")
print(f"   X_train       : {X_train.shape}  → árboles")
print(f"   X_test        : {X_test.shape}   → árboles")
print(f"   X_train_scaled: {X_train_scaled.shape}  → Logística / KNN")
print(f"   X_test_scaled : {X_test_scaled.shape}   → Logística / KNN")
print(f"   Churn rate train: {y_train.mean()*100:.1f}%")
print(f"   Churn rate test : {y_test.mean()*100:.1f}%")


### 3B — Creación de Modelos

Entrenamos **4 modelos** cubriendo los dos enfoques del desafío:

| Modelo | Normalización | Justificación |
|---|---|---|
| **Regresión Logística** | ✅ Sí | Sensible a escala; optimiza coeficientes |
| **KNN** | ✅ Sí | Basado en distancias; escala afecta proximidad |
| **Árbol de Decisión** | ❌ No | Divide por umbrales; escala irrelevante |
| **Random Forest** | ❌ No | Ensemble de árboles; invariante a escala |


In [ ]:
# ── Definición de modelos ─────────────────────────────────────────────────────
modelo_lr  = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42, C=1.0)
modelo_knn = KNeighborsClassifier(n_neighbors=7, metric='euclidean')
modelo_dt  = DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42)
modelo_rf  = RandomForestClassifier(n_estimators=200, max_depth=10,
                                     class_weight='balanced', random_state=42, n_jobs=-1)

# ── Entrenamiento ─────────────────────────────────────────────────────────────
print("Entrenando modelos...\n")

# Modelos que usan datos normalizados
modelo_lr.fit(X_train_scaled, y_train)
print("✅ Regresión Logística  — entrenado con X_train_scaled")

modelo_knn.fit(X_train_scaled, y_train)
print("✅ KNN (k=7)            — entrenado con X_train_scaled")

# Modelos que NO requieren normalización
modelo_dt.fit(X_train, y_train)
print("✅ Árbol de Decisión    — entrenado con X_train (sin escalar)")

modelo_rf.fit(X_train, y_train)
print("✅ Random Forest        — entrenado con X_train (sin escalar)")

print("\n🏁 Todos los modelos entrenados exitosamente")


### 3C — Evaluación de los Modelos

Evaluamos cada modelo con: **Accuracy, Precision, Recall, F1-Score, ROC-AUC** y **Matriz de Confusión**.

Luego realizamos un análisis crítico comparando rendimiento y detectando posibles overfitting/underfitting.


In [ ]:
# ── Función de evaluación ────────────────────────────────────────────────────
def evaluar_modelo(nombre, modelo, X_tr, X_te, y_tr, y_te):
    y_pred  = modelo.predict(X_te)
    y_proba = modelo.predict_proba(X_te)[:, 1]
    y_pred_tr = modelo.predict(X_tr)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_f1 = cross_val_score(modelo, X_tr, y_tr, cv=skf, scoring='f1', n_jobs=-1)

    return {
        'nombre'     : nombre,
        'accuracy'   : accuracy_score(y_te, y_pred),
        'precision'  : precision_score(y_te, y_pred),
        'recall'     : recall_score(y_te, y_pred),
        'f1'         : f1_score(y_te, y_pred),
        'roc_auc'    : roc_auc_score(y_te, y_proba),
        'acc_train'  : accuracy_score(y_tr, y_pred_tr),
        'cv_f1_mean' : cv_f1.mean(),
        'cv_f1_std'  : cv_f1.std(),
        'y_pred'     : y_pred,
        'y_proba'    : y_proba
    }

# ── Evaluar todos ─────────────────────────────────────────────────────────────
resultados = {}
configs = [
    ('Regresión Logística', modelo_lr,  X_train_scaled, X_test_scaled),
    ('KNN (k=7)',           modelo_knn, X_train_scaled, X_test_scaled),
    ('Árbol de Decisión',   modelo_dt,  X_train,        X_test),
    ('Random Forest',       modelo_rf,  X_train,        X_test),
]

for nombre, modelo, Xtr, Xte in configs:
    res = evaluar_modelo(nombre, modelo, Xtr, Xte, y_train, y_test)
    resultados[nombre] = res
    print(f"✅ {nombre:<22} | Acc={res['accuracy']:.3f}  F1={res['f1']:.3f}  AUC={res['roc_auc']:.3f}  CV-F1={res['cv_f1_mean']:.3f}(±{res['cv_f1_std']:.3f})")


In [ ]:
# ── Tabla comparativa completa ────────────────────────────────────────────────
tabla = pd.DataFrame({
    nombre: {
        'Accuracy'      : f"{v['accuracy']:.4f}",
        'Acc (Train)'   : f"{v['acc_train']:.4f}",
        'Precision'     : f"{v['precision']:.4f}",
        'Recall'        : f"{v['recall']:.4f}",
        'F1-Score'      : f"{v['f1']:.4f}",
        'ROC-AUC'       : f"{v['roc_auc']:.4f}",
        'CV F1 (media)' : f"{v['cv_f1_mean']:.4f}",
        'CV F1 (±std)'  : f"{v['cv_f1_std']:.4f}",
    }
    for nombre, v in resultados.items()
}).T

print("=" * 75)
print("          TABLA COMPARATIVA DE MÉTRICAS — TODOS LOS MODELOS")
print("=" * 75)
print(tabla.to_string())
print("=" * 75)

# Detectar overfitting
print("\n🔍 ANÁLISIS DE OVERFITTING/UNDERFITTING:")
for nombre, v in resultados.items():
    diff = v['acc_train'] - v['accuracy']
    if diff > 0.10:
        estado = f"⚠️  POSIBLE OVERFITTING  (train={v['acc_train']:.3f} vs test={v['accuracy']:.3f}, Δ={diff:.3f})"
    elif v['accuracy'] < 0.70:
        estado = f"⚠️  POSIBLE UNDERFITTING (accuracy test={v['accuracy']:.3f})"
    else:
        estado = f"✅ Buen balance         (Δ train-test={diff:.3f})"
    print(f"  {nombre:<22}: {estado}")


In [ ]:
# ── Gráfico comparativo de métricas ──────────────────────────────────────────
metricas_graf = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
etiquetas     = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
nombres       = list(resultados.keys())
x             = np.arange(len(metricas_graf))
width         = 0.2
colors_bar    = ['#3498DB', '#E67E22', '#E74C3C', '#2ECC71']

fig, ax = plt.subplots(figsize=(14, 6))
for i, (nombre, color) in enumerate(zip(nombres, colors_bar)):
    vals = [resultados[nombre][m] for m in metricas_graf]
    ax.bar(x + i*width, vals, width, label=nombre, color=color, alpha=0.85, edgecolor='white')

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(etiquetas, fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('3C — Comparación de Métricas por Modelo', fontsize=13, fontweight='bold')
ax.axhline(0.80, color='gray', lw=1, linestyle='--', alpha=0.6)
ax.text(4.85, 0.81, 'Umbral 0.80', fontsize=8, color='gray')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# ── Matrices de Confusión ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (nombre, v) in zip(axes, resultados.items()):
    cm   = confusion_matrix(y_test, v['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{nombre}\nF1={v["f1"]:.3f} | AUC={v["roc_auc"]:.3f}',
                 fontsize=9, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.suptitle('3C — Matrices de Confusión', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# ── Curvas ROC ────────────────────────────────────────────────────────────────
colors_roc = ['#3498DB', '#E67E22', '#E74C3C', '#2ECC71']
fig, ax = plt.subplots(figsize=(8, 6))

for (nombre, v), color in zip(resultados.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, v['y_proba'])
    ax.plot(fpr, tpr, lw=2.5, color=color, label=f'{nombre} (AUC={v["roc_auc"]:.3f})')

ax.plot([0,1],[0,1],'k--', lw=1.5, label='Aleatorio (AUC=0.50)')
ax.set_xlabel('Tasa de Falsos Positivos (FPR)', fontsize=11)
ax.set_ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=11)
ax.set_title('3C — Curvas ROC — Comparación de Modelos', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ── Classification Report del mejor modelo ────────────────────────────────────
mejor_nombre = max(resultados, key=lambda x: resultados[x]['roc_auc'])
mejor        = resultados[mejor_nombre]

print(f"🏆 MEJOR MODELO: {mejor_nombre}")
print(f"   ROC-AUC : {mejor['roc_auc']:.4f}")
print(f"   F1-Score: {mejor['f1']:.4f}")
print()
print("=" * 55)
print("CLASSIFICATION REPORT — MEJOR MODELO")
print("=" * 55)
print(classification_report(y_test, mejor['y_pred'], target_names=['No Churn', 'Churn']))


---
## 🔑 Sección 4 — Interpretación y Conclusiones

- **4A** — Análisis de importancia de variables (por tipo de modelo)
- **4B** — Conclusión estratégica con factores de cancelación y recomendaciones


### 4A — Análisis de la Importancia de las Variables

Analizamos la relevancia de cada variable según el tipo de modelo:
- **Regresión Logística** → coeficientes
- **KNN** → análisis de vecinos / correlación con error
- **Random Forest** → feature importance (reducción de impureza)
- **Árbol de Decisión** → feature importance


In [ ]:
# ── 4A.1 Regresión Logística — Coeficientes ──────────────────────────────────
coef = pd.Series(modelo_lr.coef_[0], index=X_train.columns)
top_coef = pd.concat([coef.nlargest(10), coef.nsmallest(10)]).sort_values()

bar_colors_coef = ['#E74C3C' if v > 0 else '#3498DB' for v in top_coef.values]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(range(len(top_coef)), top_coef.values, color=bar_colors_coef, edgecolor='white', height=0.7)
ax.set_yticks(range(len(top_coef)))
ax.set_yticklabels(top_coef.index, fontsize=9)
ax.axvline(0, color='gray', lw=0.8, linestyle='--')
ax.set_xlabel('Coeficiente', fontsize=11)
ax.set_title('4A — Regresión Logística: Coeficientes de las Variables\n🔴 Aumentan probabilidad de Churn  |  🔵 Reducen probabilidad de Churn',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📌 Interpretación:")
print("  Coeficiente POSITIVO → la variable AUMENTA la probabilidad de cancelación.")
print("  Coeficiente NEGATIVO → la variable REDUCE  la probabilidad de cancelación.")
print(f"\n  Top 3 variables que MÁS AUMENTAN el Churn:")
for feat, val in coef.nlargest(3).items():
    print(f"    {feat}: +{val:.4f}")
print(f"\n  Top 3 variables que MÁS REDUCEN el Churn:")
for feat, val in coef.nsmallest(3).items():
    print(f"    {feat}: {val:.4f}")


In [ ]:
# ── 4A.2 Random Forest — Feature Importance ──────────────────────────────────
imp_rf = pd.Series(modelo_rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
imp_dt = pd.Series(modelo_dt.feature_importances_, index=X_train.columns).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
colors_rf = plt.cm.YlOrRd(np.linspace(0.3, 0.9, 20))[::-1]

# Random Forest
axes[0].barh(range(20), imp_rf.head(20).values[::-1], color=colors_rf, edgecolor='white', height=0.7)
axes[0].set_yticks(range(20))
axes[0].set_yticklabels(imp_rf.head(20).index[::-1], fontsize=9)
axes[0].set_xlabel('Importancia (reducción de impureza)', fontsize=10)
axes[0].set_title('🌲 Random Forest\nTop 20 Variables Importantes', fontsize=11, fontweight='bold')

# Árbol de Decisión
colors_dt = plt.cm.Blues(np.linspace(0.3, 0.9, 20))[::-1]
axes[1].barh(range(20), imp_dt.head(20).values[::-1], color=colors_dt, edgecolor='white', height=0.7)
axes[1].set_yticks(range(20))
axes[1].set_yticklabels(imp_dt.head(20).index[::-1], fontsize=9)
axes[1].set_xlabel('Importancia (reducción de impureza)', fontsize=10)
axes[1].set_title('🌳 Árbol de Decisión\nTop 20 Variables Importantes', fontsize=11, fontweight='bold')

plt.suptitle('4A — Importancia de Variables (Modelos Basados en Árboles)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\n🔺 TOP 10 — Random Forest:")
for i, (feat, val) in enumerate(imp_rf.head(10).items(), 1):
    barra = '█' * int(val * 300)
    print(f"  {i:2d}. {feat:<45} {val:.4f}  {barra}")


In [ ]:
# ── 4A.3 KNN — Análisis de Impacto de Variables (correlación con error) ────────
# Para KNN no hay coeficientes directos. Analizamos importancia via permutation
from sklearn.inspection import permutation_importance

perm_knn = permutation_importance(
    modelo_knn, X_test_scaled, y_test,
    n_repeats=10, random_state=42, scoring='f1', n_jobs=-1
)

imp_knn = pd.Series(perm_knn.importances_mean, index=X_test_scaled.columns).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
colors_knn = ['#E74C3C' if v > 0 else '#95A5A6' for v in imp_knn.values]
ax.barh(range(len(imp_knn)), imp_knn.values[::-1], color=colors_knn[::-1], edgecolor='white', height=0.7)
ax.set_yticks(range(len(imp_knn)))
ax.set_yticklabels(imp_knn.index[::-1], fontsize=9)
ax.set_xlabel('Reducción de F1 al permutar la variable', fontsize=10)
ax.set_title('4A — KNN: Importancia de Variables (Permutation Importance)\nCuanto mayor el valor, más depende el modelo de esa variable',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()


### 4B — Conclusión Estratégica

---

#### 🏆 Rendimiento de los Modelos


In [ ]:
# Tabla dinámica con valores reales
print("=" * 80)
print("  RESUMEN COMPARATIVO DE MODELOS — TelecomX Churn Prediction")
print("=" * 80)
print(f"  {'Modelo':<25} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8} {'ROC-AUC':>9} {'CV-F1':>8}")
print("-" * 80)
for nombre, v in resultados.items():
    flag = ' 🏆' if nombre == mejor_nombre else '   '
    print(f"  {nombre+flag:<28} {v['accuracy']:>8.4f} {v['precision']:>10.4f} {v['recall']:>8.4f} {v['f1']:>8.4f} {v['roc_auc']:>9.4f} {v['cv_f1_mean']:>8.4f}")
print("=" * 80)
print(f"\n  🏆 Mejor modelo: {mejor_nombre}  (ROC-AUC={mejor['roc_auc']:.4f})")


---

#### 🔑 Principales Factores que Influyen en la Cancelación

Basándonos en los coeficientes de la Regresión Logística y la importancia de variables del Random Forest, los factores más relevantes son:

**🔺 Variables que AUMENTAN el riesgo de Churn:**

1. **Contrato mensual (`Contract_Month-to-month`)** → Es el predictor más fuerte. Clientes sin compromiso contractual cancelan con una frecuencia 3–4x mayor que los de contrato anual. Sin ataduras contractuales, la barrera de salida es mínima.

2. **Servicio de internet por Fibra Óptica (`InternetService_Fiber optic`)** → Paradójicamente, usuarios de fibra presentan mayor churn. Posibles causas: precio elevado, expectativas de calidad no satisfechas, o alta competencia en este segmento.

3. **Facturación sin papel (`PaperlessBilling_Yes`)** → Asociado a mayor churn; estos clientes tienden a ser más digitales y comparativos, con mayor disposición a cambiar de proveedor.

4. **Pago por cheque electrónico (`PaymentMethod_Electronic check`)** → Clientes que no tienen pago automático configurado muestran menor compromiso y mayor tasa de cancelación.

5. **Alto cargo mensual (`MonthlyCharges`)** → Clientes con facturas elevadas son más sensibles al precio y más propensos a buscar alternativas más económicas.

**🔻 Variables que REDUCEN el riesgo de Churn:**

1. **Alta antigüedad (`tenure`)** → El mejor predictor negativo. Clientes con más de 24 meses rara vez cancelan. La lealtad se construye con el tiempo.

2. **Contrato de 2 años (`Contract_Two year`)** → Reduce drásticamente la probabilidad de cancelación. El compromiso contractual es el escudo más efectivo.

3. **Tener Seguridad Online (`OnlineSecurity_Yes`)** → Clientes con servicios de valor añadido perciben más beneficios y son menos propensos a irse.

4. **Pago automático (débito/tarjeta)** → El pago automático reduce la fricción del proceso y está asociado a mayor retención.

5. **Tener pareja o dependientes** → Hogares con familia tienden a ser más estables y a necesitar continuidad del servicio.

---

#### 💡 Estrategias de Retención Recomendadas

| # | Estrategia | Variables relacionadas | Impacto esperado |
|---|---|---|---|
| 1 | **Campaña de migración contractual** — Ofrecer 15-20% de descuento en los primeros 6 meses para migrar de mensual a anual | Contract | Alto ⬆⬆ |
| 2 | **Programa de onboarding intensivo (primeros 3 meses)** — Llamadas de seguimiento, tutoriales y asistente dedicado | tenure | Alto ⬆⬆ |
| 3 | **Auditoría de calidad en Fibra Óptica** — Revisar SLA, tiempos de respuesta y satisfacción del cliente fibra | InternetService | Alto ⬆⬆ |
| 4 | **Incentivo por domiciliación bancaria** — Descuento o puntos por activar pago automático | PaymentMethod | Medio ⬆ |
| 5 | **Bundles de servicios a precio preferencial** — Paquetizar seguridad online, backup y soporte técnico | OnlineSecurity, TechSupport | Medio ⬆ |
| 6 | **Sistema de alertas tempranas con el modelo ML** — Scoring mensual para identificar clientes en riesgo y activar retención proactiva | Todos | Alto ⬆⬆ |

---

#### 📌 Próximos Pasos Técnicos

- **Optimización de hiperparámetros** con `GridSearchCV` o `Optuna`
- **Análisis SHAP** para explicabilidad individual cliente por cliente
- **Pipeline de producción** con reentrenamiento periódico y monitoreo de data drift
- **Segmentación por nivel de riesgo** (alto / medio / bajo) para campañas diferenciadas


In [ ]:
# ── Resumen ejecutivo final ───────────────────────────────────────────────────
print("=" * 65)
print("  RESUMEN EJECUTIVO — TelecomX Parte 2 | Alura LATAM 2026")
print("=" * 65)
print(f"  Dataset           : {df_raw.shape[0]:,} clientes, {df_raw.shape[1]} variables originales")
print(f"  Churn rate        : {df['Churn'].mean()*100:.1f}%  ({df['Churn'].sum():,} cancelaciones)")
print(f"  Dataset balanceado: {len(df_balanced):,} registros (50/50)")
print(f"  Features finales  : {X.shape[1]} columnas tras One-Hot Encoding")
print()
print("  MODELOS ENTRENADOS:")
for nombre, v in resultados.items():
    flag = ' 🏆 MEJOR' if nombre == mejor_nombre else ''
    print(f"    {nombre:<25} ROC-AUC={v['roc_auc']:.4f}  F1={v['f1']:.4f}{flag}")
print()
print("  TOP 5 FACTORES DE RIESGO (Random Forest):")
for i, (feat, val) in enumerate(imp_rf.head(5).items(), 1):
    print(f"    {i}. {feat} (importancia={val:.4f})")
print()
print("  ✅ Pipeline completo — Listo para producción")
print("=" * 65)
